# Parking V0 development baseline
**PARKING-V0-DEV-CPU-2026-10-29-r1**

Use a **CPU runtime**. No GPU, CUDA toolkit or old M67 runtime is required. This notebook freezes the reviewed **actual-camera adapter epoch 4** as V0, then scores exactly **512 development images** from the frozen nuScenes benchmark. It does not train V1, select another checkpoint, open held-out image/GT shards, export a phone model or change the app.

Outputs are raw metre errors plus prediction coverage. This is GT-oracle object-surface depth, not detection mAP, cuboid quality or ultrasonic nearest-surface distance. No per-image scale fitting or ±10 cm pass/fail rule. The old 0.716 m image-macro result is not the new metric.

**Resume:** keep the same RUN_ID and runtime. Rerun from the top after a reset. Completed prediction ZIPs are verified and reused; only an unfinished data shard repeats. Stop the previous process before restarting. Runtime/source/weight changes are rejected: preserve the run and use a new RUN_ID, never edit its registration.

## 1 Mount Drive and update main
The benchmark preparation must already be complete. The adapter result ZIP stays in its completed training folder. Edit paths below only if your completed folders have different names. All inputs remain read-only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import collections, datetime, hashlib, json, os, shlex, subprocess, sys
REVISION = 'PARKING-V0-DEV-CPU-2026-10-29-r1'
RUN_ID = 'parking_v0_actual_camera_epoch4_dev_r1'
if not RUN_ID or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_ID): raise ValueError('Invalid RUN_ID')
BASE = Path('/content/drive/MyDrive')
MOBILE = Path('/content/mobile_adas3d')
BENCHMARK = BASE/'mobile_adas3d_outputs/parking_v0_v1_benchmark/parking_v0_v1_frozen_benchmark_r1'
BUNDLE = BASE/'mobile_adas3d_outputs/parking_shared_adapter/shared_adapter_frozen_fullpool_r1/parking_shared_adapter_results.zip'
OUTPUT = BASE/'mobile_adas3d_outputs/parking_v0_dev'/RUN_ID
LOCAL = Path('/content')/('parking_v0_cache_'+RUN_ID)
SOURCE = Path('/content/Urban_Parking_V0_Source')
WEIGHTS = Path('/content/parking_v0_weights')
DEPS = Path('/content')/('parking_v0_deps_'+RUN_ID)
PYTHON = sys.executable
OUTPUT.mkdir(parents=True, exist_ok=True)
def run(command, name, cwd=None, task_env=True):
    command = [str(x) for x in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = OUTPUT/'colab_logs'/f'{name}_{stamp}.log'; log.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = ''
    env['YOLO_AUTOINSTALL'] = 'false'
    if task_env: env['PYTHONPATH'] = str(DEPS)+os.pathsep+str(MOBILE)
    print('+', shlex.join(command), '\nDurable log:', log, flush=True)
    tail = collections.deque(maxlen=30)
    with log.open('w') as stream:
        p = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in p.stdout:
                print(line,end='',flush=True); stream.write(line); stream.flush(); tail.append(line.rstrip())
        finally: p.stdout.close()
        code = p.wait(); stream.write(f'\nExit code: {code}\n')
    if code: raise RuntimeError(f'Exit {code}; preserve run and inspect {log}\n'+'\n'.join(tail))
    return log
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.exists(): run(['git','clone',url,MOBILE], 'clone', task_env=False)
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve other branches')
run(['git','pull','--ff-only'], 'update_main', MOBILE, task_env=False)
SCRIPT = MOBILE/'scripts/evaluate_parking_v0_dev.py'
for path in [SCRIPT, BENCHMARK/'benchmark_manifest.json', BUNDLE]:
    if not path.is_file(): raise FileNotFoundError(f'Required completed input is missing: {path}')
print(REVISION, '\nRead-only benchmark:', BENCHMARK, '\nRead-only V0 bundle:', BUNDLE, '\nOutput:', OUTPUT)

## 2 Prepare task-local CPU dependencies and test
Use the Colab PyTorch installation on CPU. Pin image preprocessing and Ultralytics in a separate folder; do not reinstall CUDA or modify completed runtime receipts. The native model source and pretrained Urban checkpoint are pinned and checked before use. First run may download the small Urban source/weights, not the full dataset. All processing is done in child processes, so a notebook-kernel restart is not needed.

In [ ]:
run([PYTHON,'-m','pip','install','--quiet','--no-deps','--upgrade','--target',DEPS,
     'ultralytics==8.4.61','ultralytics-thop==2.0.18','numpy==2.1.3','pillow==11.3.0',
     'opencv-python-headless==5.0.0.93','scipy==1.16.3','ijson==3.4.0.post0'], 'cpu_dependencies', MOBILE, task_env=False)
probe = "import torch,cv2,numpy,PIL,scipy,ijson; from importlib.metadata import version; assert version('ultralytics')=='8.4.61'; assert cv2.__version__=='5.0.0'; print('CPU runtime:',torch.__version__,cv2.__version__,numpy.__version__,PIL.__version__)"
run([PYTHON,'-X','faulthandler','-c',probe], 'cpu_preflight', MOBILE)
for pattern in ['test_parking_v0_dev.py','test_parking_v0_v1_benchmark.py']:
    run([PYTHON,'-m','unittest','discover','-s','tests','-p',pattern,'-v'], 'tests_'+pattern, MOBILE)
print('CPU tests passed. No training or benchmark model inference has started yet.')

## 3 Freeze and evaluate V0 on development only
This cell runs inference, not training. It verifies the selected checkpoint and original model source snapshots, records the current CPU runtime and frozen state hashes, then stages only development shards into local storage. Predictions are saved atomically in batches of at most 32 images. The unchanged registration protects against mixing results from another runtime.

Expect progress after each data shard. If interrupted, the current unfinished shard may repeat. Completed prediction batches remain on Drive. Never delete the completed training run or benchmark to repair a runtime mismatch.

In [ ]:
run([PYTHON,'-X','faulthandler','-u',SCRIPT,
     '--benchmark',BENCHMARK,'--bundle',BUNDLE,'--output',OUTPUT,'--local',LOCAL,
     '--source',SOURCE,'--weights',WEIGHTS,'--device','cpu'], 'v0_dev_evaluation', MOBILE)
REPORT = OUTPUT/'v0_dev_results.json'
result = json.loads(REPORT.read_text())
assert result['complete'] and result['role']=='dev' and result['metrics']['images']==512
assert result['model_weights_and_bn_unchanged'] and result['optimizer_steps']==0
assert result['heldout_images_opened']==0 and not result['heldout_evaluated'] and not result['v1_trained']
print('Primary conditional four-cell MAE (m):',result['metrics']['conditional_equal_four_cell_mae_m'])
for name, cell in result['metrics']['cells'].items():
    print(name, 'macro MAE:',cell['conditional_log_track_observation_macro_mae_m'],
          'coverage:',cell['prediction_coverage'],'tracks:',cell['reference_instance_tracks'],'logs:',cell['reference_recording_logs'])

## 4 Return the results
Download the ZIP below and send it for review. It contains the exact V0 registration, development metrics and point predictions, not camera images. Do not start V1 training or score held-out data until we review the baseline and define the next recipe.

In [ ]:
RESULTS = OUTPUT/'parking_v0_dev_results.zip'
if not RESULTS.is_file(): raise FileNotFoundError('Evaluation is incomplete; rerun section 3 with the same RUN_ID')
print('Return:',RESULTS,'Bytes:',RESULTS.stat().st_size)
from google.colab import files
files.download(str(RESULTS))